# Notebook 01 — Data Acquisition & Preprocessing

**Project:** Explainable Exoplanet Transit Classification  
**Output:** `kepler_gaf_dataset.npz` — train/val/test splits of 64×64 GAF images ready for ViT-B/16

### Pipeline
```
NASA Exoplanet Archive  →  KOI catalogue (labels)
Lightkurve / MAST       →  raw PDCSAP flux per KOI
                        →  normalize + phase-fold + resample
pyts                    →  Gramian Angular Field (1D → 64×64 image)
scikit-learn            →  stratified 70 / 15 / 15 split
numpy .npz              →  saved dataset
```

### Fallback path
If Lightkurve downloads are too slow, add the Mendeley dataset as a Kaggle input:  
Dataset DOI: `10.17632/wctcv34962.3`  
Then jump straight to **Section 4** (GAF conversion).

## Section 1 — Install & Imports

In [ ]:
# Kaggle: internet must be ON (Settings → Internet → On)
!pip install lightkurve pyts tqdm astropy -q

In [ ]:
import numpy as np
import pandas as pd
import requests
import warnings
from pathlib import Path
from tqdm.auto import tqdm

import lightkurve as lk
from pyts.image import GramianAngularField
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print("All imports OK")

## Section 2 — Configuration

All tunable constants in one place.

In [ ]:
CFG = {
    # Sequence & image
    'phase_bins'     : 1024,   # points after phase-folding and binning
    'gaf_size'       : 64,     # output GAF image size (64×64)
    'sigma_clip'     : 5,      # outlier rejection threshold (σ)

    # Split ratios
    'test_size'      : 0.15,
    'val_size'       : 0.15,
    'random_seed'    : 42,

    # Paths (Kaggle: /kaggle/working is writable)
    'data_dir'       : Path('/kaggle/working/data'),
    'cache_dir'      : Path('/kaggle/working/cache/lightcurves'),
}

CFG['data_dir'].mkdir(parents=True, exist_ok=True)
CFG['cache_dir'].mkdir(parents=True, exist_ok=True)

print("Config:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

## Section 3 — KOI Catalogue

Download the NASA Kepler Objects of Interest cumulative table from the NASA Exoplanet Archive.  
This gives us labels (`koi_disposition`) and orbital parameters needed for phase-folding.

In [ ]:
KOI_URL = (
    "https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI"
    "?table=cumulative&format=csv"
)

CATALOGUE_PATH = CFG['data_dir'] / 'koi_cumulative.csv'

def download_koi_catalogue(url: str, save_path: Path) -> pd.DataFrame:
    if save_path.exists():
        print(f"Using cached catalogue: {save_path}")
        return pd.read_csv(save_path, comment='#')
    print("Downloading KOI catalogue from NASA Exoplanet Archive...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    save_path.write_bytes(r.content)
    print(f"Saved ({save_path.stat().st_size / 1024:.0f} KB)")
    return pd.read_csv(save_path, comment='#')

koi_df = download_koi_catalogue(KOI_URL, CATALOGUE_PATH)
print(f"\nTotal KOIs: {len(koi_df)}")
print(koi_df['koi_disposition'].value_counts())

In [ ]:
# Keep only CONFIRMED and FALSE POSITIVE — drop CANDIDATE (uncertain label)
TARGET_COLS = [
    'kepid', 'kepoi_name', 'koi_disposition', 'koi_period',
    'koi_time0bk', 'koi_duration', 'koi_depth',
    'koi_prad', 'koi_srad', 'koi_steff',
]

binary_df = (
    koi_df[koi_df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]
    [TARGET_COLS]
    .dropna(subset=['koi_period', 'koi_time0bk'])
    .reset_index(drop=True)
    .copy()
)

# Binary label: 1 = CONFIRMED planet, 0 = FALSE POSITIVE
binary_df['label'] = (binary_df['koi_disposition'] == 'CONFIRMED').astype(int)

print(f"Binary dataset: {len(binary_df)} samples")
print(binary_df['koi_disposition'].value_counts())
print(f"Class balance: {binary_df['label'].mean():.1%} confirmed (positive class)")

In [ ]:
# Class distribution plot
counts = binary_df['koi_disposition'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=['#2196F3', '#F44336'])
ax.bar_label(bars, padding=3)
ax.set_title('KOI Class Distribution (binary subset)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(CFG['data_dir'] / 'class_distribution.png', dpi=120)
plt.show()

## Section 4 — Light Curve Download via Lightkurve

For each KOI:
1. Search all available Kepler quarters
2. Download PDCSAP flux (pipeline-corrected, systematics removed)
3. Stitch quarters into one light curve
4. Clean: remove NaNs, sigma-clip outliers, normalize
5. Phase-fold using known orbital period and reference epoch
6. Bin to fixed length (`phase_bins`)
7. Rescale to [−1, 1] for GAF
8. Cache as `.npy` so interrupted runs can resume

> **Kaggle tip:** This cell takes several hours for the full dataset. Run overnight on a CPU notebook, then reload on a GPU notebook for training.

In [ ]:
def preprocess_light_curve(
    kepid: int,
    period: float,
    epoch: float,
    cache_dir: Path,
    n_bins: int = 1024,
    sigma: int = 5,
) -> np.ndarray | None:
    """
    Download, preprocess, and phase-fold one Kepler light curve.
    Returns a float32 array of shape (n_bins,) in range [-1, 1],
    or None if the download failed.
    """
    cache_file = cache_dir / f"{kepid}.npy"
    if cache_file.exists():
        arr = np.load(cache_file)
        if len(arr) == n_bins:  # guard against stale cache with different n_bins
            return arr

    try:
        search = lk.search_lightcurve(f'KIC {kepid}', mission='Kepler', cadence='long')
        if len(search) == 0:
            return None

        lc = search.download_all(flux_column='pdcsap_flux').stitch()
        lc = lc.remove_nans().remove_outliers(sigma=sigma).normalize()

        folded = lc.fold(period=period, epoch_time=epoch)
        binned = folded.bin(n_bins=n_bins)

        flux = np.nan_to_num(binned.flux.value, nan=0.0).astype(np.float32)

        # Rescale to [-1, 1] — required by pyts GramianAngularField
        f_min, f_max = flux.min(), flux.max()
        if f_max - f_min < 1e-8:  # flat line: skip
            return None
        flux = 2.0 * (flux - f_min) / (f_max - f_min) - 1.0

        np.save(cache_file, flux)
        return flux

    except Exception:
        return None

In [ ]:
# --- Test on a single KOI before running the full batch ---
sample = binary_df.iloc[0]
flux_test = preprocess_light_curve(
    kepid=int(sample['kepid']),
    period=sample['koi_period'],
    epoch=sample['koi_time0bk'],
    cache_dir=CFG['cache_dir'],
    n_bins=CFG['phase_bins'],
)

if flux_test is not None:
    print(f"Shape: {flux_test.shape}, range: [{flux_test.min():.3f}, {flux_test.max():.3f}]")
    plt.figure(figsize=(10, 3))
    plt.plot(flux_test, lw=0.8)
    plt.title(f"Phase-folded light curve — KIC {int(sample['kepid'])} ({sample['koi_disposition']})")
    plt.xlabel('Phase bin')
    plt.ylabel('Normalised flux')
    plt.tight_layout()
    plt.show()
else:
    print("Download failed for sample KOI.")

In [ ]:
# --- Full batch download ---
# Already-cached KOIs are skipped automatically.
# Safe to interrupt and re-run.

sequences   = []
valid_idx   = []
failed_ids  = []

for idx, row in tqdm(binary_df.iterrows(), total=len(binary_df), desc='Downloading'):
    flux = preprocess_light_curve(
        kepid=int(row['kepid']),
        period=row['koi_period'],
        epoch=row['koi_time0bk'],
        cache_dir=CFG['cache_dir'],
        n_bins=CFG['phase_bins'],
        sigma=CFG['sigma_clip'],
    )
    if flux is not None:
        sequences.append(flux)
        valid_idx.append(idx)
    else:
        failed_ids.append(int(row['kepid']))

valid_df = binary_df.loc[valid_idx].reset_index(drop=True)

print(f"\nSuccessful: {len(sequences)} / {len(binary_df)}")
print(f"Failed:     {len(failed_ids)}")
print(f"Class balance after download: {valid_df['label'].mean():.1%} confirmed")

### Fallback: Mendeley preprocessed dataset

If Lightkurve downloads are too slow or hit rate limits, use the preprocessed dataset from Macedo & Zalewski (2024).  
On Kaggle: add it as a dataset input from the sidebar, then load it here.

```python
# Uncomment and adjust path to match the Mendeley dataset structure
# MENDELEY_PATH = Path('/kaggle/input/kepler-exoplanet/...') 
# sequences = [...]   # load from Mendeley files
# valid_df  = [...]   # match kepids to binary_df labels
```

Then continue from **Section 5** (GAF conversion) below.

## Section 5 — GAF Image Generation

Convert each 1D phase-folded sequence to a 2D Gramian Angular Field image.  
The GAF encodes temporal correlations as a matrix of angular cosine products — preserving the shape of the transit dip in a format the ViT can process.

Reference: Choudhary et al. 2025 validated this exact transformation on Kepler data.

In [ ]:
print(f"Converting {len(sequences)} sequences → {CFG['gaf_size']}×{CFG['gaf_size']} GAF images...")

gaf_transform = GramianAngularField(image_size=CFG['gaf_size'], method='summation')

X_raw = np.stack(sequences)              # (N, phase_bins)
X     = gaf_transform.fit_transform(X_raw).astype(np.float32)  # (N, 64, 64)
y     = valid_df['label'].values.astype(np.int64)

print(f"X: {X.shape}  dtype={X.dtype}")
print(f"y: {y.shape}  confirmed={y.mean():.1%}")

In [ ]:
# Visualise a few samples from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for cls, cls_name, row_axes in zip([1, 0], ['CONFIRMED', 'FALSE POSITIVE'], axes):
    indices = np.where(y == cls)[0][:5]
    for ax, idx in zip(row_axes, indices):
        ax.imshow(X[idx], cmap='viridis', origin='lower', vmin=-1, vmax=1)
        ax.set_title(cls_name, fontsize=8)
        ax.axis('off')
fig.suptitle('Sample GAF Images (top: CONFIRMED, bottom: FALSE POSITIVE)', fontsize=11)
plt.tight_layout()
plt.savefig(CFG['data_dir'] / 'sample_gaf_images.png', dpi=120)
plt.show()

## Section 6 — Stratified Train / Val / Test Split

70 % train · 15 % val · 15 % test.  
Stratified to preserve class balance in every split.

In [ ]:
# Step 1: carve off test set
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y,
    test_size=CFG['test_size'],
    stratify=y,
    random_state=CFG['random_seed'],
)

# Step 2: split remainder into train + val
val_fraction = CFG['val_size'] / (1.0 - CFG['test_size'])
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv,
    test_size=val_fraction,
    stratify=y_tv,
    random_state=CFG['random_seed'],
)

for name, X_s, y_s in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    print(f"{name:5s}: {len(X_s):4d} samples  |  confirmed: {y_s.mean():.1%}")

## Section 7 — Save Dataset

In [ ]:
DATASET_PATH = CFG['data_dir'] / 'kepler_gaf_dataset.npz'
METADATA_PATH = CFG['data_dir'] / 'valid_kois.csv'

np.savez_compressed(
    DATASET_PATH,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)
valid_df.to_csv(METADATA_PATH, index=False)

size_mb = DATASET_PATH.stat().st_size / (1024 ** 2)
print(f"Saved dataset:  {DATASET_PATH}  ({size_mb:.1f} MB)")
print(f"Saved metadata: {METADATA_PATH}")

## Section 8 — Verification

In [ ]:
data = np.load(DATASET_PATH)

print("Dataset verification")
print("-" * 45)
for split in ['train', 'val', 'test']:
    X_s = data[f'X_{split}']
    y_s = data[f'y_{split}']
    print(
        f"{split:5s}  X={str(X_s.shape):16s}  "
        f"dtype={X_s.dtype}  "
        f"confirmed={y_s.mean():.1%}  "
        f"range=[{X_s.min():.2f}, {X_s.max():.2f}]"
    )

print()
print("Ready for Notebook 02 — Model Evaluation")